# Chapter 28 — Debugging AI Research

**Book alignment:** Debugging AI From First Principles, Chapter 28

**Question this notebook isolates:** A generated "related work" paragraph cites three
papers. Does resolving every claim to a retrieved byte separate **H1** (fabricated
reference — no source byte), **H2** (misattributed source — resolves but contradicts the
claim), and **H3** (synthesis overreach — atoms resolve, the conclusion exceeds them)? And
what happens to a synthesis built on defective rows?

In [ ]:
# a retrieval index: doc_id -> (title, supporting_bytes)   (what an INDEPENDENT lookup returns)
INDEX = {
    "doc-07": ("Write-through invalidation at scale", "we evaluate write-through cache invalidation ... recall +2.1% (section 4)"),
}
# the generated paragraph, as numbered claims: id -> (claim, citation_string, doc_id_or_None)
CLAIMS = {
    "C1": ("segment-cached invalidation improves recall 12%", "Doe 2023", "doc-07"),   # resolves, but says +2.1% write-through
    "C2": ("segment-cached invalidation is the standard baseline", "Smith 2021", None), # no retrievable byte
    "C3": ("therefore adopt segment-cached invalidation as the proven baseline", "synthesis of C1+C2", None),
}

## 1. Resolve every reference to a retrieved byte

In [ ]:
def resolve(claim_id):
    claim, cite, doc = CLAIMS[claim_id]
    if claim_id == "C3":
        return "derived"
    if doc is None or doc not in INDEX:
        return "H1 fabricated-reference (no source byte under independent lookup)"
    title, byte_range = INDEX[doc]
    # does the quoted byte support the claim's number AND direction?
    supports = "12%" in claim and "12%" in byte_range
    if not supports:
        return f"H2 misattributed (doc says: {byte_range!r})"
    return "OK (supported)"

verdicts = {cid: resolve(cid) for cid in CLAIMS}
for cid, v in verdicts.items():
    print(f"{cid}: {v}")
assert verdicts["C1"].startswith("H2")
assert verdicts["C2"].startswith("H1")

## 2. Derive the synthesis row — a defective dependency makes it UNKNOWN

In [ ]:
deps = ["C1", "C2"]
defective = [d for d in deps if not verdicts[d].startswith("OK")]
c3_verdict = "UNKNOWN (blocked)" if defective else "SUPPORTED"
print(f"C3 depends on {deps}; defective dependencies: {defective}")
print(f"C3 verdict: {c3_verdict}")
assert c3_verdict.startswith("UNKNOWN")
print("a conclusion built on unresolved rows is unresolved - never 'partially supported', never 'probably fine'")

## 3. Delete every row without a hash — what remains is the research

In [ ]:
research = [cid for cid, v in verdicts.items() if v.startswith("OK")]
draft = [cid for cid, v in verdicts.items() if not v.startswith("OK") and cid != "C3"]
print("supported (keep):", research)
print("draft (fabricated / misattributed):", draft)
assert research == [] and set(draft) == {"C1", "C2"}
print("\ntypography is not provenance; a second generation agreeing is the same failure mode sampled twice")

## What we earned

Chapter 3's evidence hygiene, aimed at prose about the world. Every factual claim was
numbered and resolved to a retrieved byte via an *independent* lookup: C1 **misattributed**
(the source reports +2.1% write-through, not 12% segment-cached), C2 **fabricated** (no
retrievable byte), and the synthesis C3 therefore **UNKNOWN** — eloquence does not
rehabilitate a conclusion built on defective rows. A citation without a source hash is a
rumor with a bibliography.

**Notebook 29 / Chapter 29** takes the agent that consumed this research and looped: the
trajectory itself becomes the patient.